<a href="https://colab.research.google.com/github/j0seph-gp/road-accident-black-spot-analysis/blob/main/Road_Accident_Black_Spot_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import platform
import subprocess

print("Operating System:")
print(platform.platform())

print("\nJava version:")
result = subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
)
print(result.stderr)

Operating System:
Linux-6.6.122+-x86_64-with-glibc2.39

Java version:
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)



CHECKING HADOOP INSTALLED OR NOT ?


In [2]:
!hadoop version

/bin/bash: line 1: hadoop: command not found


Installing Hadoop

In [3]:
!wget -q https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz
!tar -xzf hadoop-3.3.6.tar.gz
!mv hadoop-3.3.6 /content/hadoop

print("Hadoop files downloaded and extracted successfully.")

Hadoop files downloaded and extracted successfully.


STEP 5 — Set Hadoop environment variables

Now we need to tell Colab where Hadoop is installed.

In [4]:
import os

os.environ["HADOOP_HOME"] = "/content/hadoop"
os.environ["HADOOP_CONF_DIR"] = "/content/hadoop/etc/hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["PATH"]

print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print("HADOOP_CONF_DIR =", os.environ["HADOOP_CONF_DIR"])

HADOOP_HOME = /content/hadoop
HADOOP_CONF_DIR = /content/hadoop/etc/hadoop


STEP 6 — Verify Hadoop installation


In [5]:
!$HADOOP_HOME/bin/hadoop version

ERROR: JAVA_HOME is not set and could not be found.


STEP 6 — Set JAVA_HOME

Run only this code in a new Colab cell:

In [6]:
import os

java_home = "/usr/lib/jvm/java-21-openjdk-amd64"

os.environ["JAVA_HOME"] = java_home
os.environ["PATH"] = (
    os.environ["JAVA_HOME"] + "/bin:"
    + os.environ["HADOOP_HOME"] + "/bin:"
    + os.environ["PATH"]
)

print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64


In [7]:
!$JAVA_HOME/bin/java -version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)


In [8]:
!$HADOOP_HOME/bin/hadoop version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /content/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


In [9]:
import os

hadoop_env = "/content/hadoop/etc/hadoop/hadoop-env.sh"

with open(hadoop_env, "r") as f:
    content = f.read()

java_line = 'export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64'

if "export JAVA_HOME=" in content:
    lines = content.splitlines()
    lines = [
        java_line if line.strip().startswith("export JAVA_HOME=") else line
        for line in lines
    ]
    content = "\n".join(lines) + "\n"
else:
    content += "\n" + java_line + "\n"

with open(hadoop_env, "w") as f:
    f.write(content)

print("JAVA_HOME configured in hadoop-env.sh")

JAVA_HOME configured in hadoop-env.sh


In [10]:
!$HADOOP_HOME/bin/hadoop version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /content/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


In [11]:
!$HADOOP_HOME/bin/hdfs version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.003s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hadoop 3.3.6
Source code repository https://github.com/apache/hadoop.git -r 1be78238728da9266a4f88195058f08fd012bf9c
Compiled by ubuntu on 2023-06-18T08:22Z
Compiled on platform linux-x86_64
Compiled with protoc 3.7.1
From source with checksum 5652179ad55f76cb287d9c633bb53bbd
This command was run using /content/hadoop/share/hadoop/common/hadoop-common-3.3.6.jar


In [12]:
import os

hadoop_conf = "/content/hadoop/etc/hadoop"

core_site = """<?xml version="1.0"?>
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>
"""

with open(f"{hadoop_conf}/core-site.xml", "w") as f:
    f.write(core_site)

print("core-site.xml configured successfully.")

core-site.xml configured successfully.


In [13]:
import os

hadoop_conf = "/content/hadoop/etc/hadoop"

hdfs_site = """<?xml version="1.0"?>
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>

    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///content/hadoop_data/namenode</value>
    </property>

    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///content/hadoop_data/datanode</value>
    </property>
</configuration>
"""

with open(f"{hadoop_conf}/hdfs-site.xml", "w") as f:
    f.write(hdfs_site)

print("hdfs-site.xml configured successfully.")

hdfs-site.xml configured successfully.


In [14]:
!$HADOOP_HOME/bin/hdfs namenode -format -force

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-09-12 06:32:53,116 INFO namenode.NameNode: STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = cab95c258577/172.28.0.12
STARTUP_MSG:   args = [-format, -force]
STARTUP_MSG:   version = 3.3.6
STARTUP_MSG:   classpath = /content/hadoop/etc/hadoop:/content/hadoop/share/hadoop/common/lib/hadoop-annotations-3.3.6.jar:/content/hadoop/share/hadoop/common/lib/jersey-core-1.19.4.jar:/content/hadoop/share/hadoop/common/lib/kerb-crypto-1.0.1.jar:/content/hadoop/share/hadoop/common/lib/commons-beanutils-1.9.4.jar:/content/hadoop/share/hadoop/common/lib/slf4j-api-1.7.36.jar:/content/hadoop/share

STEP 14 — Start HDFS

Run only this cell:

In [15]:
!$HADOOP_HOME/sbin/start-dfs.sh

Starting namenodes on [[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
localhost]
ERROR: Attempting to operate on hdfs namenode as root
ERROR: but there is no HDFS_NAMENODE_USER defined. Aborting operation.
Starting datanodes
ERROR: Attempting to operate on hdfs datanode as root
ERROR: but there is no HDFS_DATANODE_USER defined. Aborting operation.
Starting journal nodes [[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate]
ERROR: Attempt

In [16]:
import os

hadoop_env = "/content/hadoop/etc/hadoop/hadoop-env.sh"

with open(hadoop_env, "r") as f:
    content = f.read()

settings = """
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
export HDFS_JOURNALNODE_USER=root
export HDFS_ZKFC_USER=root
"""

if "HDFS_NAMENODE_USER=root" not in content:
    content += settings

with open(hadoop_env, "w") as f:
    f.write(content)

print("HDFS user configuration added successfully.")

HDFS user configuration added successfully.


In [17]:
!$HADOOP_HOME/sbin/start-dfs.sh

Starting namenodes on [[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
localhost]
ERROR: JAVA_HOME is not set and could not be found.
Starting datanodes
ERROR: JAVA_HOME is not set and could not be found.
Starting journal nodes [[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate]
ERROR: JAVA_HOME is not set and could not be found.


In [18]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["HADOOP_HOME"] = "/content/hadoop"
os.environ["HADOOP_CONF_DIR"] = "/content/hadoop/etc/hadoop"

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])

JAVA_HOME = /usr/lib/jvm/java-21-openjdk-amd64
HADOOP_HOME = /content/hadoop


In [19]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

os.system(
    "export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && "
    "/content/hadoop/sbin/start-dfs.sh"
)

768

In [20]:
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2356 Jps


In [21]:
!grep -E "JAVA_HOME|HDFS_.*_USER" /content/hadoop/etc/hadoop/hadoop-env.sh

#  JAVA_HOME=/usr/java/testing hdfs dfs -ls
# Technically, the only required environment variable is JAVA_HOME.
# export JAVA_HOME=
# export HDFS_DATANODE_SECURE_USER=hdfs
# export HDFS_NFS3_SECURE_USER=nfsserver
# export HDFS_NAMENODE_USER=hdfs
export HDFS_NAMENODE_USER=root
export HDFS_DATANODE_USER=root
export HDFS_SECONDARYNAMENODE_USER=root
export HDFS_JOURNALNODE_USER=root
export HDFS_ZKFC_USER=root


In [22]:
hadoop_env = "/content/hadoop/etc/hadoop/hadoop-env.sh"

with open(hadoop_env, "r") as f:
    content = f.read()

# Remove any existing active JAVA_HOME line
lines = [
    line for line in content.splitlines()
    if not line.strip().startswith("export JAVA_HOME=")
]

# Add the correct Java path
lines.append("export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64")

with open(hadoop_env, "w") as f:
    f.write("\n".join(lines) + "\n")

print("JAVA_HOME added to hadoop-env.sh")

JAVA_HOME added to hadoop-env.sh


In [23]:
!grep "export JAVA_HOME=" /content/hadoop/etc/hadoop/hadoop-env.sh

# export JAVA_HOME=
export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64


In [24]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HADOOP_CONF_DIR=/content/hadoop/etc/hadoop && \
/content/hadoop/sbin/start-dfs.sh

Starting namenodes on [[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
localhost]
sed: -e expression #1, char 7: unknown option to `s'
[0.001s][warning][os,container]: hostname contains invalid characters
Cgroup: ssh: Could not resolve hostname cgroup: Name or service not known
at: ssh: Could not resolve hostname at: No address associated with hostname
to: ssh: Could not resolve hostname to: No address associated with hostname
path: ssh: Could not resolve hostname path: Name or service not known
seems: ssh: Could not resolve hostname seems: Name or service not known
controller: ssh: Could not resolve hostname controller: Name or service not known
memory: ssh: Could not resolve hostname memory: Name or service 

In [25]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

!$HADOOP_HOME/bin/hdfs --daemon start namenode

In [26]:
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2929 NameNode
2966 Jps


In [27]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

!$HADOOP_HOME/bin/hdfs --daemon start datanode

In [28]:
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
3025 DataNode
2929 NameNode
3061 Jps


In [29]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -ls /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [30]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -ls / ; \
echo "Exit code: $?"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Exit code: 0


In [31]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -mkdir -p /road_accidents

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [32]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -ls /

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
drwxr-xr-x   - root supergroup          0 2026-09-12 06:33 /road_accidents


In [33]:
!ls -lh /content

total 697M
drwxr-xr-x 11 ubuntu ubuntu 4.0K Sep 12 06:32 hadoop
-rw-r--r--  1 root   root   697M Jun 25  2023 hadoop-3.3.6.tar.gz
drwxr-xr-x  4 root   root   4.0K Sep 12 06:33 hadoop_data
drwxr-xr-x  1 root   root   4.0K Sep  4 13:25 sample_data


In [34]:
from google.colab import files

uploaded = files.upload()

Saving accident_dataset.csv to accident_dataset.csv


In [35]:
import pandas as pd

df = pd.read_csv("/content/accident_dataset.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Rows: 3500
Columns: 11

Column names:
['latitude', 'longitude', 'hour', 'temperature', 'visibility', 'traffic_density', 'weather', 'road_type', 'weekend', 'peak_hour', 'risk_score']

First 5 rows:


,latitude,longitude,hour,temperature,visibility,traffic_density,weather,road_type,weekend,peak_hour,risk_score
0,13.014856,77.553069,14,14.2,2,2,0,1,0,0,0.7520
1,13.039986,77.612158,19,27.4,1,0,0,2,0,1,0.5782
2,12.905146,77.742477,10,36.7,1,2,2,1,1,1,0.9800
3,12.953085,77.545456,4,40.4,2,2,0,0,0,0,0.8665
4,12.916933,77.603683,0,15.2,2,0,0,0,0,0,0.7057


In [36]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -put -f /content/accident_dataset.csv /road_accidents/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate


In [37]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -ls -h /road_accidents/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
-rw-r--r--   1 root supergroup    159.9 K 2026-09-12 06:34 /road_accidents/accident_dataset.csv


In [38]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -cat /road_accidents/accident_dataset.csv | head -5

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
latitude,longitude,hour,temperature,visibility,traffic_density,weather,road_type,weekend,peak_hour,risk_score
13.014856,77.553069,14,14.2,2,2,0,1,0,0,0.752
13.039986,77.612158,19,27.4,1,0,0,2,0,1,0.5782
cat: Unable to write to output stream.


In [39]:
!hive --version

/bin/bash: line 1: hive: command not found


In [40]:
!wget -q https://archive.apache.org/dist/hive/hive-3.1.3/apache-hive-3.1.3-bin.tar.gz
!tar -xzf apache-hive-3.1.3-bin.tar.gz
!mv apache-hive-3.1.3-bin /content/hive

print("Hive downloaded and extracted successfully.")

Hive downloaded and extracted successfully.


In [41]:
import os

os.environ["HIVE_HOME"] = "/content/hive"
os.environ["PATH"] = (
    os.environ["HIVE_HOME"] + "/bin:"
    + os.environ["PATH"]
)

print("HIVE_HOME =", os.environ["HIVE_HOME"])

HIVE_HOME = /content/hive


In [42]:
!$HIVE_HOME/bin/hive --version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive 3.1.3
Git git://MacBook-Pro.fios-router.home/Users/ngangam/commit/hive -r 4df4d75bf1e16fe0af75aad0b4179c34c07fc975
Compiled by ngangam on Sun Apr 3 16:58:16 EDT 2022
From source with checksu

In [43]:
!mkdir -p /content/hive_warehouse
!mkdir -p /content/hive_tmp

print("Hive warehouse and temporary directories created.")

Hive warehouse and temporary directories created.


In [44]:
hive_conf = "/content/hive/conf/hive-site.xml"

hive_site = """<?xml version="1.0" encoding="UTF-8"?>
<configuration>

    <property>
        <name>javax.jdo.option.ConnectionURL</name>
        <value>jdbc:derby:;databaseName=/content/hive_metastore_db;create=true</value>
    </property>

    <property>
        <name>hive.metastore.warehouse.dir</name>
        <value>hdfs://localhost:9000/user/hive/warehouse</value>
    </property>

    <property>
        <name>hive.exec.scratchdir</name>
        <value>hdfs://localhost:9000/tmp/hive</value>
    </property>

</configuration>
"""

with open(hive_conf, "w") as f:
    f.write(hive_site)

print("Hive configuration created successfully.")

Hive configuration created successfully.


In [45]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -mkdir -p /user/hive/warehouse && \
/content/hadoop/bin/hdfs dfs -mkdir -p /tmp/hive

print("Hive HDFS directories created successfully.")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Hive HDFS directories created successfully.


In [46]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
/content/hadoop/bin/hdfs dfs -ls /user/hive/

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Found 1 items
drwxr-xr-x   - root supergroup          0 2026-09-12 06:34 /user/hive/warehouse


In [47]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
export HIVE_HOME=/content/hive && \
/content/hive/bin/hive -e "SHOW DATABASES;"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 4a576757-9c70-4b3e-82a0-587f6c080059
Exception in thread "main" java.lang.ClassCastException: class jdk.internal.loader.ClassLoaders$AppClassLoader cannot be cast to class java.

In [48]:
!ls -d /usr/lib/jvm/* | grep -E "java-11|jdk-11"

In [49]:
!apt-get update -qq
!apt-get install -y openjdk-11-jdk -qq

print("Java 11 installation completed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../00-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../01-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../02-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1.1build1) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../03-session-migration_0.3.9build1_amd64.deb ...
Unpacking session-migration (0.3.9build1) ...
Selecting previously unse

In [50]:
! /usr/lib/jvm/java-11-openjdk-amd64/bin/java -version

openjdk version "11.0.32" 2026-07-21
OpenJDK Runtime Environment (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)


In [51]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

print("JAVA_HOME =", os.environ["JAVA_HOME"])

JAVA_HOME = /usr/lib/jvm/java-11-openjdk-amd64


In [52]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HIVE_HOME=/content/hive && \
/content/hive/bin/hive -e "SHOW DATABASES;"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 13da7e80-2ba4-4ce9-9ae9-389f224f906a
Exception in thread "main" java.lang.ClassCastException: class jdk.internal.loader.ClassLoaders$AppClassLoader cannot be cast to class java.

In [53]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
echo "JAVA_HOME=$JAVA_HOME" && \
$JAVA_HOME/bin/java -version && \
/content/hive/bin/hive --version

JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
openjdk version "11.0.32" 2026-07-21
OpenJDK Runtime Environment (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of 

In [54]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export PATH=$JAVA_HOME/bin:$PATH && \
which java && \
java -version && \
echo "Hive JAVA_HOME: $JAVA_HOME"

/usr/lib/jvm/java-11-openjdk-amd64/bin/java
openjdk version "11.0.32" 2026-07-21
OpenJDK Runtime Environment (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)
Hive JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [55]:
!grep -E "HADOOP_HOME|JAVA_HOME|HIVE_AUX_JARS_PATH" /content/hive/bin/hive | head -20

if [ -d "${HIVE_AUX_JARS_PATH}" ]; then
  hive_aux_jars_abspath=`cd ${HIVE_AUX_JARS_PATH} && pwd`
elif [ "${HIVE_AUX_JARS_PATH}" != "" ]; then 
  HIVE_AUX_JARS_PATH=`echo $HIVE_AUX_JARS_PATH | sed 's/,/:/g'`
      HIVE_AUX_JARS_PATH=`cygpath -p -w "$HIVE_AUX_JARS_PATH"`
      HIVE_AUX_JARS_PATH=`echo $HIVE_AUX_JARS_PATH | sed 's/;/,/g'`
  AUX_CLASSPATH=${AUX_CLASSPATH}:${HIVE_AUX_JARS_PATH}
  AUX_PARAM="file://$(echo ${HIVE_AUX_JARS_PATH} | sed 's/:/,file:\/\//g')"
# supress the HADOOP_HOME warnings in 1.x.x
export HADOOP_HOME_WARN_SUPPRESS=true 
# HADOOP_HOME env variable overrides hadoop in the path
HADOOP_HOME=${HADOOP_HOME:-${HADOOP_PREFIX:-$HADOOP_DIR}}
if [ "$HADOOP_HOME" == "" ]; then
  echo "Cannot find hadoop installation: \$HADOOP_HOME or \$HADOOP_PREFIX must be set or hadoop must be in the path";
for f in ${HADOOP_HOME}/share/hadoop/tools/lib/hadoop-distcp-*.jar; do
HADOOP=$HADOOP_HOME/bin/hadoop
  echo "Cannot find hadoop installation: \$HADOOP_HOME or \$HADOOP_PREFIX must 

In [56]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
echo "JAVA_HOME=$JAVA_HOME" && \
echo "HADOOP_HOME=$HADOOP_HOME" && \
echo "JAVA=$(which java)" && \
echo "JAVA_VERSION:" && \
java -version

JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64
HADOOP_HOME=/content/hadoop
JAVA=/usr/lib/jvm/java-11-openjdk-amd64/bin/java
JAVA_VERSION:
openjdk version "11.0.32" 2026-07-21
OpenJDK Runtime Environment (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 11.0.32+9-post-1ubuntu1-24.04-Ubuntu, mixed mode, sharing)


In [57]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/schematool -dbType derby -info

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Metastore connection URL:	 jdbc:derby:;databaseName=/content/hive_metastore_db;create=true
Metastore Connection Driver :	 org.apache.derby.jdbc.EmbeddedDriver
Metastore connection User:	 APP
org.

In [58]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/schematool -dbType derby -initSchema

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Metastore connection URL:	 jdbc:derby:;databaseName=/content/hive_metastore_db;create=true
Metastore Connection Driver :	 org.apache.derby.jdbc.EmbeddedDriver
Metastore connection User:	 APP
Star

In [59]:
!rm -rf /content/hive_metastore_db

In [60]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/schematool -dbType derby -initSchema

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Metastore connection URL:	 jdbc:derby:;databaseName=/content/hive_metastore_db;create=true
Metastore Connection Driver :	 org.apache.derby.jdbc.EmbeddedDriver
Metastore connection User:	 APP
Star

In [61]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/schematool -dbType derby -initSchema

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Metastore connection URL:	 jdbc:derby:;databaseName=/content/hive_metastore_db;create=true
Metastore Connection Driver :	 org.apache.derby.jdbc.EmbeddedDriver
Metastore connection User:	 APP
Star

In [62]:
!ls -lah /content/hive_metastore_db

total 44K
drwxr-xr-x 5 root root 4.0K Sep 12 06:36 .
drwxr-xr-x 1 root root 4.0K Sep 12 06:36 ..
-rw-r--r-- 1 root root    4 Sep 12 06:36 dbex.lck
-rw-r--r-- 1 root root   38 Sep 12 06:36 db.lck
drwxr-xr-x 2 root root 4.0K Sep 12 06:36 log
-rw-r--r-- 1 root root  608 Sep 12 06:36 README_DO_NOT_TOUCH_FILES.txt
drwxr-xr-x 2 root root  12K Sep 12 06:36 seg0
-rw-r--r-- 1 root root  905 Sep 12 06:36 service.properties
drwxr-xr-x 2 root root 4.0K Sep 12 06:36 tmp


In [63]:
!jps

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
3025 DataNode
2929 NameNode
6350 Jps


In [64]:
!rm -rf /content/hive_metastore_db
!ls -ld /content/hive_metastore_db

ls: cannot access '/content/hive_metastore_db': No such file or directory


In [65]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/schematool -dbType derby -initSchema

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Metastore connection URL:	 jdbc:derby:;databaseName=/content/hive_metastore_db;create=true
Metastore Connection Driver :	 org.apache.derby.jdbc.EmbeddedDriver
Metastore connection User:	 APP
Star

In [66]:
!export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "SHOW DATABASES;"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 5dfb4dda-4657-4d20-818a-7699396d00a8
Exception in thread "main" java.lang.ClassCastException: class jdk.internal.loader.ClassLoaders$AppClassLoader cannot be cast to class java.

In [67]:
!ls -d /usr/lib/jvm/* | grep -E "java-8|openjdk-8"

In [68]:
!apt-get update -qq
!apt-get install -y openjdk-8-jdk

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libgail-common libgail18t64 libgtk2.0-0t64 libgtk2.0-bin libgtk2.0-common
  librsvg2-common openjdk-8-jdk-headless openjdk-8-jre openjdk-8-jre-headless
Suggested packages:
  gvfs openjdk-8-demo openjdk-8-source visualvm libnss-mdns fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libgail-common libgail18t64 libgtk2.0-0t64 libgtk2.0-bin libgtk2.0-common
  librsvg2-common openjdk-8-jdk openjdk-8-jdk-headless openjdk-8-jre
  openjdk-8-jre-headless
0 upgraded, 10 newly installed, 0 to remove and 30 not upgraded.
Need to get 46.0 MB of archives.
After t

In [69]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export PATH=$JAVA_HOME/bin:$PATH && \
java -version

openjdk version "1.8.0_502"
OpenJDK Runtime Environment (build 1.8.0_502-8u502-ga~us1-0ubuntu1~24.04-b07)
OpenJDK 64-Bit Server VM (build 25.502-b07, mixed mode)


In [70]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "SHOW DATABASES;"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 679c6fcf-b25b-4f4f-834a-3d95a06bc9b4
Exception in thread "main" java.lang.ClassCastException: class jdk.internal.loader.ClassLoaders$AppClassLoader cannot be cast to class java.

In [71]:
!grep -n "JAVA_HOME" /content/hadoop/etc/hadoop/hadoop-env.sh

37:#  JAVA_HOME=/usr/java/testing hdfs dfs -ls
47:# Technically, the only required environment variable is JAVA_HOME.
54:# export JAVA_HOME=
437:export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64


In [72]:
!sed -i 's|export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64|export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64|' /content/hadoop/etc/hadoop/hadoop-env.sh

In [73]:
!grep -n "export JAVA_HOME" /content/hadoop/etc/hadoop/hadoop-env.sh

54:# export JAVA_HOME=
437:export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64


In [74]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "SHOW DATABASES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = dbcd3a1d-d0e8-4875-b89e-7784cf725d00

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Exception in thread "main" java.lang.RuntimeException: The dir: hdfs://localhost:9000/tmp/hive on HDFS should be writable. Current permissions are: rwxr-xr-x
	at org.apache.hadoop.hive.ql.exec.Utilities.ensurePathIsWritable(Utilities.java:4501)
	at org.apache.hadoop.hive.ql.session.SessionState.createRootHDFSDir(SessionState.java:760)
	at

In [75]:
!export HADOOP_HOME=/content/hadoop && \
export PATH=$HADOOP_HOME/bin:$PATH && \
hdfs dfs -chmod 777 /tmp/hive

In [76]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "SHOW DATABASES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 17c0a92d-2926-4884-a140-2fea9f96daa9

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = 33564215-fc7e-4b10-b309-c12db5f144af
OK
default
Time taken: 1.375 seconds, Fetched: 1 row(s)


In [77]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "CREATE DATABASE IF NOT EXISTS road_accident_db; SHOW DATABASES;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = cb643538-4ddb-4de9-9bcf-ff526f29c26b

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = abf88c72-5473-4a4b-9efd-15322ff31d0a
OK
Time taken: 1.374 seconds
OK
default
road_accident_db
Time taken: 0.305 seconds, Fetched: 2 row(s)


In [78]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && export HADOOP_HOME=/content/hadoop && export HIVE_HOME=/content/hive && export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && /content/hive/bin/hive -e "USE road_accident_db; CREATE EXTERNAL TABLE IF NOT EXISTS accident_data (latitude DOUBLE, longitude DOUBLE, hour INT, temperature DOUBLE, visibility INT, traffic_density INT, weather INT, road_type INT, weekend INT, peak_hour INT, risk_score DOUBLE) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' STORED AS TEXTFILE LOCATION 'hdfs://localhost:9000/road_accidents';"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = a3e58619-0d9b-45fd-aea0-62f0ee6cc38b

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = 9a67f18c-70e7-48da-b5b4-8e60e5c56930
OK
Time taken: 1.377 seconds
OK
Time taken: 1.277 seconds


In [79]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && export HADOOP_HOME=/content/hadoop && export HIVE_HOME=/content/hive && export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && /content/hive/bin/hive -e "USE road_accident_db; CREATE EXTERNAL TABLE IF NOT EXISTS accident_data (latitude DOUBLE, longitude DOUBLE, hour INT, temperature DOUBLE, visibility INT, traffic_density INT, weather INT, road_type INT, weekend INT, peak_hour INT, risk_score DOUBLE) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' STORED AS TEXTFILE LOCATION 'hdfs://localhost:9000/road_accidents';"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 695e4954-9a0e-4f3d-afc3-441191ed3b44

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = 2f60eb59-5e14-440b-a550-112ce4862d0f
OK
Time taken: 1.031 seconds
OK
Time taken: 0.604 seconds


In [80]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "USE road_accident_db; SELECT * FROM accident_data LIMIT 5;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = a336dcba-b990-4055-9454-06249c6991d5

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = 2e7c50d8-7153-4561-91f8-79e6f6b49bf8
OK
Time taken: 1.051 seconds
OK
NULL	NULL	NULL	NULL	NULL	NULL	NULL	NULL	NULL	NULL	NULL
13.014856	77.553069	14	14.2	2	2	0	1	0	0	0.752
13.039986	77.612158	19	27.4	1	0	0	2	0	1	0.5782
12.905146	77.742477	10	36.7	1	2	2	1	1	1	0.98
12.953085	77.545456	4	40.4	2	2	0	0	0	0	0.8665
Time taken: 3.

In [81]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "USE road_accident_db; SELECT COUNT(*) FROM accident_data;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 5a7481de-15d2-40bb-ae40-f7ddba532b3e

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = 53495391-9124-4465-8214-4ec47b681346
OK
Time taken: 1.118 seconds
Query ID = root_20260912063948_2dfcb4b3-11c9-4c8c-b7cd-06ddd04bb84e
Total jobs = 1
Launching Job 1 out of 1
Number of reduce tasks determined at compile time: 1
In order to change the average load for a reducer (in bytes):
  set hive.exec.reducers.bytes.pe

In [82]:
!export JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64 && \
export HADOOP_HOME=/content/hadoop && \
export HIVE_HOME=/content/hive && \
export PATH=$JAVA_HOME/bin:$HADOOP_HOME/bin:$HIVE_HOME/bin:$PATH && \
/content/hive/bin/hive -e "USE road_accident_db; CREATE OR REPLACE VIEW accident_clean AS SELECT * FROM accident_data WHERE latitude IS NOT NULL AND longitude IS NOT NULL; SELECT COUNT(*) FROM accident_clean;"

SLF4J: Class path contains multiple SLF4J bindings.
SLF4J: Found binding in [jar:file:/content/hive/lib/log4j-slf4j-impl-2.17.1.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: Found binding in [jar:file:/content/hadoop/share/hadoop/common/lib/slf4j-reload4j-1.7.36.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See http://www.slf4j.org/codes.html#multiple_bindings for an explanation.
SLF4J: Actual binding is of type [org.apache.logging.slf4j.Log4jLoggerFactory]
Hive Session ID = 6c581326-cc9a-4d28-8ec7-b07914bcd09c

Logging initialized using configuration in jar:file:/content/hive/lib/hive-common-3.1.3.jar!/hive-log4j2.properties Async: true
Hive Session ID = 602754bf-3639-483b-93a1-82bd48e2857f
OK
Time taken: 1.052 seconds
OK
Time taken: 3.444 seconds
Query ID = root_20260912064020_04bf1511-58e8-4505-90b1-93a74266171a
Total jobs = 1
Launching Job 1 out of 1
Number of reduce tasks determined at compile time: 1
In order to change the average load for a reducer (in bytes):
  se

In [84]:
!export JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64 && \
export PATH=$JAVA_HOME/bin:$PATH && \
java -version && \
spark-submit --version

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12" 2026-07-21
OpenJDK Runtime Environment (build 21.0.12+8-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12+8-1-24.04-Ubuntu, mixed mode, sharing)
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate

In [85]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RoadAccidentAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [87]:
accident_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("hdfs://localhost:9000/road_accidents/accident_dataset.csv")

print("Total records:", accident_df.count())
accident_df.show(5)

Total records: 3500
+---------+---------+----+-----------+----------+---------------+-------+---------+-------+---------+----------+
| latitude|longitude|hour|temperature|visibility|traffic_density|weather|road_type|weekend|peak_hour|risk_score|
+---------+---------+----+-----------+----------+---------------+-------+---------+-------+---------+----------+
|13.014856|77.553069|  14|       14.2|         2|              2|      0|        1|      0|        0|     0.752|
|13.039986|77.612158|  19|       27.4|         1|              0|      0|        2|      0|        1|    0.5782|
|12.905146|77.742477|  10|       36.7|         1|              2|      2|        1|      1|        1|      0.98|
|12.953085|77.545456|   4|       40.4|         2|              2|      0|        0|      0|        0|    0.8665|
|12.916933|77.603683|   0|       15.2|         2|              0|      0|        0|      0|        0|    0.7057|
+---------+---------+----+-----------+----------+---------------+-------+---

In [88]:
accident_df.describe(
    "hour",
    "temperature",
    "visibility",
    "traffic_density",
    "risk_score"
).show()

+-------+-----------------+------------------+------------------+------------------+-------------------+
|summary|             hour|       temperature|        visibility|   traffic_density|         risk_score|
+-------+-----------------+------------------+------------------+------------------+-------------------+
|  count|             3500|              3500|              3500|              3500|               3500|
|   mean|           11.478|26.878400000000028|             1.276|0.9585714285714285| 0.7874232000000035|
| stddev|6.925383268061449| 8.648012408452455|0.7604439070016359|0.7428794801436747|0.18326941055101176|
|    min|                0|              12.0|                 0|                 0|             0.1664|
|    max|               23|              42.0|                 2|                 2|               0.98|
+-------+-----------------+------------------+------------------+------------------+-------------------+



In [89]:
high_risk_df = accident_df.orderBy(
    accident_df["risk_score"].desc()
).limit(10)

high_risk_df.show(truncate=False)

+---------+---------+----+-----------+----------+---------------+-------+---------+-------+---------+----------+
|latitude |longitude|hour|temperature|visibility|traffic_density|weather|road_type|weekend|peak_hour|risk_score|
+---------+---------+----+-----------+----------+---------------+-------+---------+-------+---------+----------+
|13.001177|77.567933|1   |40.8       |2         |1              |1      |0        |1      |0        |0.98      |
|12.905146|77.742477|10  |36.7       |1         |2              |2      |1        |1      |1        |0.98      |
|12.908434|77.611764|10  |29.1       |1         |2              |0      |1        |0      |1        |0.98      |
|13.012439|77.598788|21  |27.9       |1         |2              |2      |0        |0      |0        |0.98      |
|13.081818|77.581635|8   |18.4       |1         |1              |2      |1        |0      |1        |0.98      |
|12.916322|77.584997|4   |19.7       |2         |1              |2      |0        |0      |0    

In [90]:
from pyspark.sql.functions import avg

risk_by_hour = accident_df.groupBy("hour") \
    .agg(avg("risk_score").alias("average_risk")) \
    .orderBy("hour")

risk_by_hour.show(24)

+----+------------------+
|hour|      average_risk|
+----+------------------+
|   0|0.8781774999999998|
|   1|0.8659600000000004|
|   2|0.8611471014492759|
|   3|0.8714622516556293|
|   4|0.8747187050359718|
|   5|0.7259111888111891|
|   6|0.6738083916083916|
|   7|0.7500496503496502|
|   8|0.8406360544217693|
|   9|0.8554209150326804|
|  10|0.8212111940298508|
|  11|0.6868061538461542|
|  12|0.6951981012658227|
|  13|0.7294339999999999|
|  14|0.7111679999999996|
|  15|0.7275538888888892|
|  16|0.7195070921985816|
|  17|0.8158107913669069|
|  18|0.8183630573248412|
|  19|0.8374056000000005|
|  20|0.8457325581395352|
|  21|0.7128527777777779|
|  22|0.7177945578231294|
|  23| 0.873586363636364|
+----+------------------+



In [91]:
from pyspark.sql.functions import round, avg, count, max

black_spots = accident_df \
    .withColumn("lat_zone", round("latitude", 3)) \
    .withColumn("lon_zone", round("longitude", 3)) \
    .groupBy("lat_zone", "lon_zone") \
    .agg(
        count("*").alias("record_count"),
        avg("risk_score").alias("average_risk"),
        max("risk_score").alias("maximum_risk")
    ) \
    .orderBy("average_risk", ascending=False) \
    .limit(20)

black_spots.show(20, truncate=False)

+--------+--------+------------+------------+------------+
|lat_zone|lon_zone|record_count|average_risk|maximum_risk|
+--------+--------+------------+------------+------------+
|12.898  |77.618  |1           |0.98        |0.98        |
|13.009  |77.563  |1           |0.98        |0.98        |
|12.993  |77.545  |1           |0.98        |0.98        |
|13.043  |77.604  |1           |0.98        |0.98        |
|12.961  |77.596  |1           |0.98        |0.98        |
|12.965  |77.596  |1           |0.98        |0.98        |
|12.98   |77.603  |2           |0.98        |0.98        |
|12.989  |77.542  |1           |0.98        |0.98        |
|12.936  |77.69   |1           |0.98        |0.98        |
|12.971  |77.598  |1           |0.98        |0.98        |
|12.909  |77.606  |1           |0.98        |0.98        |
|13.098  |77.506  |1           |0.98        |0.98        |
|12.956  |77.564  |1           |0.98        |0.98        |
|12.956  |77.61   |1           |0.98        |0.98       

In [92]:
risk_by_traffic = accident_df.groupBy("traffic_density") \
    .agg(
        count("*").alias("record_count"),
        avg("risk_score").alias("average_risk")
    ) \
    .orderBy("traffic_density")

risk_by_traffic.show()

+---------------+------------+------------------+
|traffic_density|record_count|      average_risk|
+---------------+------------+------------------+
|              0|        1041|0.6799221902017305|
|              1|        1563|0.7998916186820249|
|              2|         896|0.8905709821428619|
+---------------+------------+------------------+



In [93]:
risk_by_weather = accident_df.groupBy("weather") \
    .agg(
        count("*").alias("record_count"),
        avg("risk_score").alias("average_risk")
    ) \
    .orderBy("weather")

risk_by_weather.show()

+-------+------------+------------------+
|weather|record_count|      average_risk|
+-------+------------+------------------+
|      0|        2115|0.7082052009456264|
|      1|         505|0.8763918811881217|
|      2|         880|0.9267605681818256|
+-------+------------+------------------+



In [94]:
risk_by_road = accident_df.groupBy("road_type") \
    .agg(
        count("*").alias("record_count"),
        avg("risk_score").alias("average_risk")
    ) \
    .orderBy("road_type")

risk_by_road.show()

+---------+------------+------------------+
|road_type|record_count|      average_risk|
+---------+------------+------------------+
|        0|        1245|0.8215366265060277|
|        1|        1556|0.7795760925449906|
|        2|         699|0.7441311874105878|
+---------+------------+------------------+



In [95]:
black_spots_final = accident_df \
    .withColumn("lat_zone", round("latitude", 3)) \
    .withColumn("lon_zone", round("longitude", 3)) \
    .groupBy("lat_zone", "lon_zone") \
    .agg(
        count("*").alias("accident_records"),
        avg("risk_score").alias("average_risk"),
        max("risk_score").alias("maximum_risk")
    ) \
    .filter("accident_records >= 3") \
    .orderBy("average_risk", ascending=False)

black_spots_final.show(20, truncate=False)

+--------+--------+----------------+------------------+------------+
|lat_zone|lon_zone|accident_records|average_risk      |maximum_risk|
+--------+--------+----------------+------------------+------------+
|12.933  |77.698  |3               |0.9443666666666667|0.98        |
|13.003  |77.555  |3               |0.9354666666666667|0.98        |
|13.043  |77.596  |3               |0.9318333333333334|0.98        |
|12.96   |77.597  |3               |0.9020999999999999|0.98        |
|12.971  |77.593  |3               |0.8483999999999999|0.98        |
|13.041  |77.609  |3               |0.8462000000000001|0.9575      |
|12.995  |77.563  |3               |0.8285999999999999|0.98        |
|12.988  |77.565  |3               |0.7942666666666667|0.98        |
|12.92   |77.692  |3               |0.7919333333333333|0.98        |
|12.987  |77.543  |3               |0.7688            |0.98        |
|13.04   |77.593  |4               |0.7588            |0.98        |
|12.921  |77.688  |3              

In [96]:
black_spots_final.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("/content/black_spots_results")

In [97]:
import pandas as pd

result_file = "/content/black_spots_results"

import os
print("Files created:")
print(os.listdir(result_file))

Files created:


FileNotFoundError: [Errno 2] No such file or directory: '/content/black_spots_results'

In [98]:
import os

print(os.listdir("/content"))

['.config', 'hive', 'accident_dataset.csv', 'apache-hive-3.1.3-bin.tar.gz', 'spark-warehouse', 'derby.log', 'hive_tmp', 'hive_metastore_db', 'hadoop', 'hadoop_data', 'hadoop-3.3.6.tar.gz', 'hive_warehouse', 'sample_data']


In [99]:
print("Black-spot records:", black_spots_final.count())
black_spots_final.show(10, truncate=False)

Black-spot records: 16
+--------+--------+----------------+------------------+------------+
|lat_zone|lon_zone|accident_records|average_risk      |maximum_risk|
+--------+--------+----------------+------------------+------------+
|12.933  |77.698  |3               |0.9443666666666667|0.98        |
|13.003  |77.555  |3               |0.9354666666666667|0.98        |
|13.043  |77.596  |3               |0.9318333333333334|0.98        |
|12.96   |77.597  |3               |0.9020999999999999|0.98        |
|12.971  |77.593  |3               |0.8483999999999999|0.98        |
|13.041  |77.609  |3               |0.8462000000000001|0.9575      |
|12.995  |77.563  |3               |0.8285999999999999|0.98        |
|12.988  |77.565  |3               |0.7942666666666667|0.98        |
|12.92   |77.692  |3               |0.7919333333333333|0.98        |
|12.987  |77.543  |3               |0.7688            |0.98        |
+--------+--------+----------------+------------------+------------+
only showin

In [100]:
black_spots_final.toPandas().to_csv(
    "/content/black_spots_results.csv",
    index=False
)

print("Saved successfully!")
print("File:", "/content/black_spots_results.csv")

Saved successfully!
File: /content/black_spots_results.csv


In [101]:
import pandas as pd

black_spots_check = pd.read_csv("/content/black_spots_results.csv")

print("Number of black spots:", len(black_spots_check))
print("\nColumns:")
print(list(black_spots_check.columns))

print("\nData:")
display(black_spots_check)

Number of black spots: 16

Columns:
['lat_zone', 'lon_zone', 'accident_records', 'average_risk', 'maximum_risk']

Data:


,lat_zone,lon_zone,accident_records,average_risk,maximum_risk
0,12.933,77.698,3,0.944367,0.9800
1,13.003,77.555,3,0.935467,0.9800
2,13.043,77.596,3,0.931833,0.9800
3,12.960,77.597,3,0.902100,0.9800
4,12.971,77.593,3,0.848400,0.9800
5,13.041,77.609,3,0.846200,0.9575
6,12.995,77.563,3,0.828600,0.9800
7,12.988,77.565,3,0.794267,0.9800
8,12.920,77.692,3,0.791933,0.9800
9,12.987,77.543,3,0.768800,0.9800


In [102]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [103]:
import shutil

drive_path = "/content/drive/MyDrive/road_accident_black_spots.csv"

shutil.copy(
    "/content/black_spots_results.csv",
    drive_path
)

print("Black-spot results saved successfully!")
print("Location:", drive_path)

Black-spot results saved successfully!
Location: /content/drive/MyDrive/road_accident_black_spots.csv


In [104]:
import os

path = "/content/drive/MyDrive/road_accident_black_spots.csv"

print("File exists:", os.path.exists(path))
print("File size:", os.path.getsize(path), "bytes")

File exists: True
File size: 673 bytes


In [105]:
import shutil

dataset_drive_path = "/content/drive/MyDrive/road_accident_dataset.csv"

shutil.copy(
    "/content/accident_dataset.csv",
    dataset_drive_path
)

print("Dataset saved successfully!")
print("Location:", dataset_drive_path)

Dataset saved successfully!
Location: /content/drive/MyDrive/road_accident_dataset.csv


In [106]:
import os

dataset_path = "/content/drive/MyDrive/road_accident_dataset.csv"

print("Dataset exists:", os.path.exists(dataset_path))
print("Dataset size:", os.path.getsize(dataset_path), "bytes")

Dataset exists: True
Dataset size: 163721 bytes
